# Clase 3 — Construir una red neuronal con PyTorch

Curso **"Deep Learning con Python y PyTorch desde cero"** — DIH DATAlife (Hub de Innovación Digital Europeo), a partir de la certificación EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Duración estimada de esta clase:** 110 min

## Objetivos de aprendizaje de esta clase

- Explicar la relación entre neuronas, capas y arquitectura de una red
- Definir una red neuronal como una clase que hereda de `nn.Module`
- Usar capas `nn.Linear` y funciones de activación para construir un clasificador simple
- Ejecutar un forward pass y entender qué hace autograd por debajo

En la clase anterior aprendiste a representar y cargar datos con `Dataset` y `DataLoader`. Hoy toca definir el modelo en sí: la red neuronal que va a recibir esos lotes de imágenes del dataset MNIST y producir una predicción de qué dígito representan. Todavía no la vamos a entrenar — eso llega en la clase 4 — pero al final de esta sesión tendrás una arquitectura completa y funcional, capaz de recibir un lote real de datos y devolver una salida con sentido.

## Parte 1 — De neuronas a arquitectura

Piensa en una red neuronal como una cadena de producción organizada en niveles. Cada nivel es una **capa** (*layer*), y dentro de cada capa trabajan varias unidades pequeñas llamadas **neuronas**. Cada neurona hace algo muy simple: recibe varios números, calcula una suma ponderada (cada entrada multiplicada por un **peso**, más un **sesgo** o *bias*) y entrega un único número de salida.

Una red completa organiza sus capas en tres tipos:

- **Capa de entrada**: recibe los datos en crudo (en nuestro caso, los píxeles de una imagen ya aplanados)
- **Capas ocultas**: extraen patrones cada vez más complejos a partir de los datos
- **Capa de salida**: entrega el resultado final (en nuestro caso, una puntuación por cada dígito del 0 al 9)

A esta organización completa —cuántas capas hay, cuántas neuronas tiene cada una y cómo se conectan— la llamamos **arquitectura**. Dos redes pueden usar exactamente las mismas piezas de PyTorch y comportarse de forma muy distinta según la arquitectura elegida.

## Parte 2 — `nn.Module`: el plano de tu red

En PyTorch, toda red neuronal se define como una clase de Python que hereda de `nn.Module`. Esta clase te obliga a organizar tu red en dos bloques:

- `__init__`: aquí **declaras** las piezas de tu red (qué capas existen y de qué tamaño)
- `forward`: aquí **conectas** esas piezas (en qué orden pasan los datos por cada capa)

Dentro de `__init__`, la primera línea siempre es `super().__init__()`: llama al constructor de `nn.Module` y es imprescindible para que PyTorch registre correctamente tus capas.

Empecemos importando lo necesario y comprobando la versión de PyTorch instalada.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

print("Versión de PyTorch:", torch.__version__)
print("Versión de torchvision:", torchvision.__version__)

## Parte 3 — `nn.Linear`: capas totalmente conectadas

La pieza más básica para construir capas es `nn.Linear`. Cada neurona de una capa `nn.Linear` recibe la salida de absolutamente todas las neuronas de la capa anterior (por eso se llama **totalmente conectada**, *fully connected*, o **densa**).

`nn.Linear(in_features, out_features)` recibe dos números: cuántos valores entran (`in_features`) y cuántas neuronas tiene la capa, es decir, cuántos valores salen (`out_features`). Vamos a crear una capa suelta y a inspeccionar sus pesos y sesgos, antes de montar la red completa.

In [ ]:
# Una capa totalmente conectada: 784 valores de entrada (28x28 píxeles aplanados), 64 de salida
capa_ejemplo = nn.Linear(in_features=784, out_features=64)

print("Forma de los pesos:", capa_ejemplo.weight.shape)  # 64 neuronas x 784 entradas
print("Forma de los sesgos:", capa_ejemplo.bias.shape)   # un sesgo por neurona

# Los pesos y sesgos se inicializan al azar: por eso rompemos la simetría entre neuronas
print("\nPrimeros valores de los pesos:\n", capa_ejemplo.weight[0, :5])

## Parte 4 — Funciones de activación: por qué hacen falta

Si encadenas varias capas `nn.Linear` sin nada más entre medias, el resultado sigue siendo, matemáticamente, una única transformación lineal, sin importar cuántas capas apiles. Para que la red pueda aprender patrones complejos necesitas introducir **no linealidad** entre capas, y eso es justo lo que hace una **función de activación**.

- **ReLU** (*Rectified Linear Unit*, unidad lineal rectificada): deja pasar sin cambios los valores positivos y convierte en cero los negativos. Es la activación más habitual en capas ocultas
- **`log_softmax`**: se usa en la capa de salida de un clasificador con varias clases. Convierte las puntuaciones finales en una distribución de probabilidad: las diez salidas (una por dígito) reparten entre sí la probabilidad total

Veamos ReLU en acción sobre unos valores de ejemplo, antes de usarla dentro de la red.

In [ ]:
valores_ejemplo = torch.tensor([-3.0, -0.5, 0.0, 1.2, 4.0])
print("Valores originales:", valores_ejemplo)
print("Tras aplicar ReLU:  ", F.relu(valores_ejemplo))

## Parte 5 — Construimos el clasificador completo

Ya tienes todas las piezas: capas `nn.Linear` y funciones de activación. Vamos a definir un clasificador para MNIST que recibe una imagen aplanada de 784 valores y produce una puntuación para cada uno de los diez dígitos (0-9).

La arquitectura:

- Capa de entrada -> primera capa oculta: 784 -> 128 neuronas
- Primera capa oculta -> segunda capa oculta: 128 -> 64 neuronas
- Segunda capa oculta -> capa de salida: 64 -> 10 neuronas (una por dígito)

In [ ]:
class ClasificadorMNIST(nn.Module):
    def __init__(self):
        # Llamada obligatoria al constructor de nn.Module
        super().__init__()

        # Declaramos las capas: cada nn.Linear conecta por completo
        # la capa anterior con la siguiente
        self.fc1 = nn.Linear(784, 128)  # entrada -> primera capa oculta
        self.fc2 = nn.Linear(128, 64)   # primera capa oculta -> segunda capa oculta
        self.fc3 = nn.Linear(64, 10)    # segunda capa oculta -> salida (10 dígitos)

    def forward(self, x):
        # ReLU tras cada capa oculta para introducir no linealidad
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # La salida no lleva ReLU: usamos log_softmax para obtener
        # una distribución de probabilidad sobre las 10 clases
        x = F.log_softmax(self.fc3(x), dim=1)
        return x


# Instanciamos la red: aquí se inicializan pesos y sesgos al azar
modelo = ClasificadorMNIST()
print(modelo)

`print(modelo)` te muestra la arquitectura completa: las tres capas `nn.Linear`, con sus tamaños de entrada y salida. Fíjate en que esta representación no incluye las funciones de activación: viven dentro del método `forward`, no como una capa declarada en `__init__`.

## Parte 6 — Forward pass con un lote real de MNIST

Vamos a cargar el dataset MNIST con `torchvision`, igual que hiciste en la clase anterior, y a pasar un lote real de imágenes por la red que acabas de definir.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transformacion = transforms.ToTensor()

datos_mnist = datasets.MNIST(root="./datos", train=True, download=True, transform=transformacion)
cargador = DataLoader(datos_mnist, batch_size=16, shuffle=True)

# Tomamos un único lote del cargador
imagenes, etiquetas = next(iter(cargador))
print("Forma del lote de imágenes:", imagenes.shape)   # [16, 1, 28, 28]
print("Forma del lote de etiquetas:", etiquetas.shape)  # [16]

El lote llega con forma `[16, 1, 28, 28]`: 16 imágenes, 1 canal (escala de grises), 28x28 píxeles. Pero nuestra red espera un vector de 784 valores por imagen, no una matriz de 28x28. Hay que **aplanar** cada imagen antes de pasarla por la red.

In [ ]:
# Aplanamos cada imagen del lote: de [16, 1, 28, 28] a [16, 784]
# El -1 le dice a PyTorch "calcula tú este tamaño", aquí mantiene el número de imágenes del lote
imagenes_aplanadas = imagenes.view(-1, 784)
print("Forma tras aplanar:", imagenes_aplanadas.shape)

# El forward pass: los datos atraviesan la red y obtenemos una predicción
salida = modelo(imagenes_aplanadas)
print("Forma de la salida:", salida.shape)  # [16, 10]: una fila por imagen, 10 valores por fila

print("\nSalida para la primera imagen del lote:\n", salida[0])
print("\n¿Suman 1 las probabilidades de la primera imagen? (aproximadamente):",
      torch.exp(salida[0]).sum().item())

Los valores de `salida` son log-probabilidades (por eso usamos `torch.exp` para recuperar probabilidades normales antes de sumarlas): cuanto más alto el valor para una clase, más confianza tiene la red en que la imagen pertenece a ese dígito. Como todavía no hemos entrenado la red, estas predicciones son básicamente ruido: los pesos siguen siendo los valores aleatorios iniciales.

## Parte 7 — Qué hace autograd por debajo

Aunque no lo veas explícitamente en el código, cada operación del `forward` que acabas de ejecutar (`self.fc1(x)`, `F.relu(...)`, `F.log_softmax(...)`) ha quedado registrada por **autograd**, el motor de diferenciación automática de PyTorch. Ese registro forma lo que se llama un **grafo de cómputo**: un mapa de qué operación produjo cada tensor y a partir de qué otros tensores.

Ese grafo es exactamente lo que la clase siguiente va a necesitar: cuando calculemos cuánto se equivoca la red (la pérdida) y llamemos a `.backward()`, autograd recorrerá el grafo hacia atrás para calcular el gradiente de cada peso y cada sesgo, es decir, en qué dirección y cuánto hay que ajustarlos para reducir el error. Por ahora, solo necesitas saber que ese trabajo ya está ocurriendo de forma automática cada vez que ejecutas un `forward`; no tienes que programarlo tú.

## Resumen de la clase

- Una red neuronal se organiza en capas (entrada, ocultas, salida) formadas por neuronas con pesos y sesgos
- Toda red en PyTorch hereda de `nn.Module`, con `__init__` para declarar capas y `forward` para conectarlas
- `nn.Linear(in_features, out_features)` crea una capa totalmente conectada
- Las funciones de activación (ReLU en capas ocultas, `log_softmax` en la salida de un clasificador) introducen no linealidad y hacen interpretable el resultado
- Antes de pasar una imagen por una red de capas `nn.Linear`, hay que aplanarla con `.view(-1, tamaño)`
- El `forward` pass ya activa autograd, que registra el grafo de cómputo necesario para el entrenamiento

En la próxima clase vas a usar esta misma arquitectura para entrenarla de verdad: calcular la pérdida, propagarla hacia atrás con `.backward()` y actualizar los pesos con un optimizador.